In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

# SQLAlchemy engine instead of raw psycopg2 -- this is the "proper" way
# pandas wants a DB connection, and it silences that warning you saw earlier.
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER')}:{os.getenv('POSTGRES_PASSWORD')}"
    f"@localhost:5432/{os.getenv('POSTGRES_DB')}"
)

In [ ]:
# Query 1: Highest trade price per hour
q1 = pd.read_sql("""
    SELECT DATE_TRUNC('hour', trade_time) AS hour, MAX(price) AS highest_price
    FROM cleaned_trades
    GROUP BY DATE_TRUNC('hour', trade_time)
    ORDER BY hour
""", engine)

plt.figure(figsize=(8, 4))
plt.bar(q1['hour'].astype(str), q1['highest_price'], color='#4C72B0')
plt.title('Highest Trade Price per Hour')
plt.xlabel('Hour')
plt.ylabel('Price (USDT)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Query 2: Total volume + trade count per hour
q2 = pd.read_sql("""
    SELECT DATE_TRUNC('hour', candle_start) AS hour,
           SUM(volume) AS total_volume,
           SUM(trade_count) AS total_trades
    FROM ohlcv_candles
    GROUP BY DATE_TRUNC('hour', candle_start)
    ORDER BY hour
""", engine)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.bar(q2['hour'].astype(str), q2['total_volume'], color='#55A868', label='Volume')
ax1.set_ylabel('Total Volume (BTC)', color='#55A868')
ax1.set_xlabel('Hour')
plt.xticks(rotation=45, ha='right')

ax2 = ax1.twinx()
ax2.plot(q2['hour'].astype(str), q2['total_trades'], color='#C44E52', marker='o', label='Trade Count')
ax2.set_ylabel('Total Trade Count', color='#C44E52')

plt.title('Volume & Trade Count per Hour')
plt.tight_layout()
plt.show()

In [ ]:
# Query 3: Most volatile candles
q3 = pd.read_sql("""
    SELECT symbol, candle_start, volatility_pct
    FROM ohlcv_candles
    ORDER BY volatility_pct DESC
    LIMIT 5
""", engine)

plt.figure(figsize=(8, 4))
plt.barh(q3['candle_start'].astype(str), q3['volatility_pct'], color='#8172B2')
plt.title('Top 5 Most Volatile Candles')
plt.xlabel('Volatility (%)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Query 4: Busiest candles by trade count
q4 = pd.read_sql("""
    SELECT symbol, candle_start, trade_count
    FROM ohlcv_candles
    ORDER BY trade_count DESC
    LIMIT 5
""", engine)

plt.figure(figsize=(8, 4))
plt.barh(q4['candle_start'].astype(str), q4['trade_count'], color='#CCB974')
plt.title('Top 5 Busiest Candles (Trade Count)')
plt.xlabel('Number of Trades')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()